<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Demo - End-to-end Evaluation with MLflow

## Overview
Evaluation is one of the most important stages of the generative AI lifecycle—it defines the quality bar your system must meet. This lesson first explains *why* we evaluate GenAI systems and *which* metrics matter for the retrieval and generation components of a RAG pipeline, then puts that theory into practice: you will load the RAG model you registered earlier and evaluate it with MLflow using both built-in and custom metrics.

## Learning Objectives
By the end of this demo, you will be able to:
- Describe common evaluation metrics used for evaluating retrieval and generation models.
- Discuss the difference between performance and system metrics.
- Load a model from the model registry for evaluation.
- Identify common built-in evaluation metrics in MLflow.
- Define a custom evaluation metric with MLflow.
- Run an evaluation test and view the results using the UI or code.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, ML Environment Version 5 (ML v5)</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless compute with the ML v5 environment</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

To select the **ML v5** environment:

1. Click the **Environment** button on the right sidebar
2. Open the **Environment version** dropdown
3. Select **ML v5**

If you do not see **Serverless** available, contact your workspace administrator.

## A. Evaluating the Whole System

<div class="ar">
  <style>    @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&display=swap');
    .ar{font-family:'DM Sans','Segoe UI',Arial,sans-serif;max-width:1040px;margin:14px auto 6px;color:#1B3139;}
    .ar *{box-sizing:border-box;}
    .ar-lead{font-size:14.5px;color:#3a4d54;margin:0 0 14px;}
    .ar-zone{background:#fff;border:1px solid #EEEDE9;border-radius:14px;box-shadow:0 5px 14px rgba(27,49,57,.07);padding:16px 18px;margin-bottom:14px;}
    .ar-zt{display:inline-block;font-size:12px;font-weight:800;letter-spacing:.5px;text-transform:uppercase;color:#fff;border-radius:7px;padding:5px 12px;margin-bottom:14px;}
    .ar-flow{display:flex;flex-wrap:wrap;align-items:center;gap:10px;}
    .ar-step{flex:1 1 130px;display:flex;flex-direction:column;align-items:center;gap:8px;text-align:center;background:var(--tint);border-radius:11px;padding:14px 10px;font-size:12.5px;font-weight:700;color:#1B3139;}
    .ar-ic{width:40px;height:40px;border-radius:11px;background:#fff;color:var(--acc);display:inline-flex;align-items:center;justify-content:center;box-shadow:0 1px 4px rgba(27,49,57,.12);}
    .ar-ic svg{width:23px;height:23px;}
    .ar-arw{flex:0 0 auto;color:#9AA6A0;font-size:20px;font-weight:800;}
    .ar-side{display:flex;flex-wrap:wrap;gap:10px;margin-bottom:16px;}
    .ar-pill{background:var(--t);color:var(--c);font-weight:700;font-size:13px;border-radius:20px;padding:6px 14px;}</style>
  <div class="ar-side">
    <span class="ar-pill" style="--t:rgba(255,95,70,.10);--c:#B23A2B;">Cost metrics &mdash; resources &amp; time</span>
    <span class="ar-pill" style="--t:rgba(66,153,224,.12);--c:#2272B4;">Performance &mdash; direct &amp; indirect value</span>
    <span class="ar-pill" style="--t:rgba(0,169,114,.12);--c:#1B6E48;">Custom metrics for your use case</span>
  </div>
  <div class="ar-zone" style="--zc:#4299E0;">
    <span class="ar-zt" style="background:#4299E0;">1 · Document Embedding</span>
    <div class="ar-flow"><div class="ar-step" style="--acc:#4299E0;--tint:rgba(66,153,224,.12);"><span class="ar-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M6 3h8l4 4v14H6z"/><path d="M14 3v4h4"/><path d="M9 12h6M9 15h6"/></svg></span><span>Source documents</span></div><span class="ar-arw">&rarr;</span><div class="ar-step" style="--acc:#4299E0;--tint:rgba(66,153,224,.12);"><span class="ar-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M9 4.5a2.6 2.6 0 00-2.6 2.6 2.4 2.4 0 00-1.4 4.3A2.6 2.6 0 006 15.9a2.4 2.4 0 003 2.6 2.4 2.4 0 003-.6 2.4 2.4 0 003 .6 2.4 2.4 0 003-2.6 2.6 2.6 0 001-4.5 2.4 2.4 0 00-1.4-4.3A2.6 2.6 0 0015 4.5a2.5 2.5 0 00-3 .9 2.5 2.5 0 00-3-.9Z"/></svg></span><span>Embedding model</span></div><span class="ar-arw">&rarr;</span><div class="ar-step" style="--acc:#4299E0;--tint:rgba(66,153,224,.12);"><span class="ar-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><ellipse cx="12" cy="6" rx="7" ry="3"/><path d="M5 6v12c0 1.7 3.1 3 7 3s7-1.3 7-3V6"/><path d="M5 12c0 1.7 3.1 3 7 3s7-1.3 7-3"/></svg></span><span>Embeddings in vector store</span></div></div>
  </div>
  <div class="ar-zone" style="--zc:#FFAB00;">
    <span class="ar-zt" style="background:#C7860A;">2 · User Query &amp; Retrieval</span>
    <div class="ar-flow"><div class="ar-step" style="--acc:#FFAB00;--tint:rgba(255,171,0,.14);"><span class="ar-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><circle cx="12" cy="8" r="3.4"/><path d="M5.5 20a6.5 6.5 0 0113 0"/></svg></span><span>User asks a question</span></div><span class="ar-arw">&rarr;</span><div class="ar-step" style="--acc:#FFAB00;--tint:rgba(255,171,0,.14);"><span class="ar-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M9 4.5a2.6 2.6 0 00-2.6 2.6 2.4 2.4 0 00-1.4 4.3A2.6 2.6 0 006 15.9a2.4 2.4 0 003 2.6 2.4 2.4 0 003-.6 2.4 2.4 0 003 .6 2.4 2.4 0 003-2.6 2.6 2.6 0 001-4.5 2.4 2.4 0 00-1.4-4.3A2.6 2.6 0 0015 4.5a2.5 2.5 0 00-3 .9 2.5 2.5 0 00-3-.9Z"/></svg></span><span>Query &rarr; embedding</span></div><span class="ar-arw">&rarr;</span><div class="ar-step" style="--acc:#FFAB00;--tint:rgba(255,171,0,.14);"><span class="ar-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><circle cx="11" cy="11" r="6"/><path d="M20 20l-4.5-4.5"/></svg></span><span>Similarity search</span></div></div>
  </div>
  <div class="ar-zone" style="--zc:#00A972;">
    <span class="ar-zt" style="background:#00A972;">3 · Generation</span>
    <div class="ar-flow"><div class="ar-step" style="--acc:#00A972;--tint:rgba(0,169,114,.12);"><span class="ar-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M4 5h16v11H9l-5 4V5z"/><path d="M12 8v5M9.5 10.5h5"/></svg></span><span>Retrieved context as prompt</span></div><span class="ar-arw">&rarr;</span><div class="ar-step" style="--acc:#00A972;--tint:rgba(0,169,114,.12);"><span class="ar-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M9 4.5a2.6 2.6 0 00-2.6 2.6 2.4 2.4 0 00-1.4 4.3A2.6 2.6 0 006 15.9a2.4 2.4 0 003 2.6 2.4 2.4 0 003-.6 2.4 2.4 0 003 .6 2.4 2.4 0 003-2.6 2.6 2.6 0 001-4.5 2.4 2.4 0 00-1.4-4.3A2.6 2.6 0 0015 4.5a2.5 2.5 0 00-3 .9 2.5 2.5 0 00-3-.9Z"/></svg></span><span>Generation model</span></div><span class="ar-arw">&rarr;</span><div class="ar-step" style="--acc:#00A972;--tint:rgba(0,169,114,.12);"><span class="ar-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M20 5H4v11h5l5 4v-4h6z"/><path d="M8 10.5l2.5 2.5 4-4.5"/></svg></span><span>User receives an answer</span></div></div>
  </div>
</div>


## B. Performance Metrics

### B1. Evaluating RAG Pipeline

<div class="b1">
  <style>
    @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&display=swap');
    .b1{font-family:'DM Sans','Segoe UI',Arial,sans-serif;max-width:1040px;margin:14px auto 6px;color:#1B3139;}
    .b1 *{box-sizing:border-box;}
    .b1-row{display:flex;flex-wrap:wrap;gap:22px;align-items:center;}
    .b1-col{flex:1 1 320px;}
    .b1-list{list-style:none;margin:6px 0 0;padding:0;}
    .b1-list li{position:relative;padding-left:26px;font-size:14.5px;line-height:1.5;color:#3a4d54;margin-bottom:11px;}
    .b1-list li b{color:#1B3139;}
    .b1-list li::before{content:'';position:absolute;left:0;top:2px;width:17px;height:17px;border-radius:50%;background:#4299E0;}
    .b1-list li::after{content:'';position:absolute;left:5px;top:6px;width:4px;height:7px;border:solid #fff;border-width:0 2px 2px 0;transform:rotate(45deg);}
    .b1-sub{list-style:none;margin:8px 0 0;padding:0 0 0 26px;}
    .b1-sub li{font-size:13px;color:#5A6F77;margin-bottom:5px;}
    .b1-gauges{flex:1 1 300px;display:flex;flex-direction:column;gap:12px;}
    .g-card{display:flex;align-items:center;gap:14px;background:#fff;border:1px solid #EEEDE9;border-radius:12px;box-shadow:0 4px 12px rgba(27,49,57,.06);padding:10px 16px;}
    .g-card svg{width:96px;height:62px;flex:0 0 auto;}
    .g-card span{font-size:14.5px;font-weight:800;color:#1B3139;}
  </style>
  <div class="b1-row">
    <div class="b1-col">
      <ul class="b1-list">
        <li>Evaluate each RAG component <b>separately and together</b>.</li>
        <li>Components to evaluate:</li>
      </ul>
      <ul class="b1-sub"><li>Chunking &mdash; method and size</li><li>Embedding model</li><li>Vector store &mdash; retrieval and re-ranker</li><li>Generator</li></ul>
    </div>
    <div class="b1-gauges">
      <div class="g-card"><svg viewBox="0 0 120 78" aria-hidden="true"><path d="M18 58 A42 42 0 0 1 102 58" fill="none" stroke="#EEEDE9" stroke-width="9" stroke-linecap="round"/><path d="M18 58 A42 42 0 0 1 102 58" fill="none" stroke="#00A972" stroke-width="9" stroke-linecap="round" stroke-dasharray="95.03999999999999 999"/><line x1="60" y1="58" x2="86.8" y2="25.6" stroke="#00A972" stroke-width="3.4" stroke-linecap="round"/><circle cx="60" cy="58" r="5" fill="#00A972"/></svg><span>Chunking Performance</span></div>
      <div class="g-card"><svg viewBox="0 0 120 78" aria-hidden="true"><path d="M18 58 A42 42 0 0 1 102 58" fill="none" stroke="#EEEDE9" stroke-width="9" stroke-linecap="round"/><path d="M18 58 A42 42 0 0 1 102 58" fill="none" stroke="#FF5F46" stroke-width="9" stroke-linecap="round" stroke-dasharray="44.88 999"/><line x1="60" y1="58" x2="39.8" y2="21.2" stroke="#FF5F46" stroke-width="3.4" stroke-linecap="round"/><circle cx="60" cy="58" r="5" fill="#FF5F46"/></svg><span>Retrieval Performance</span></div>
      <div class="g-card"><svg viewBox="0 0 120 78" aria-hidden="true"><path d="M18 58 A42 42 0 0 1 102 58" fill="none" stroke="#EEEDE9" stroke-width="9" stroke-linecap="round"/><path d="M18 58 A42 42 0 0 1 102 58" fill="none" stroke="#1B3139" stroke-width="9" stroke-linecap="round" stroke-dasharray="66.0 999"/><line x1="60" y1="58" x2="60.0" y2="16.0" stroke="#1B3139" stroke-width="3.4" stroke-linecap="round"/><circle cx="60" cy="58" r="5" fill="#1B3139"/></svg><span>Generator Performance</span></div>
    </div>
  </div>
</div>

### B2. Evaluation Metrics

<div class="ov">
  <style>
    @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&display=swap');
    .ov{font-family:'DM Sans','Segoe UI',Arial,sans-serif;max-width:1040px;margin:14px auto 6px;color:#1B3139;}
    .ov *{box-sizing:border-box;}
    .ov-diag{background:linear-gradient(180deg,#FBFAF8,#F4F7F9);border:1px solid #EEEDE9;border-radius:14px;box-shadow:0 5px 14px rgba(27,49,57,.07);padding:16px;}
    .ov-diag svg{width:100%;height:auto;max-width:760px;display:block;margin:0 auto;}
  </style>
  <div class="ov-diag">
    <svg viewBox="0 0 760 360" role="img" aria-label="Evaluation metrics triangle: retrieval questions relate query and context, generation questions relate context, response and query">
      <defs><marker id="ov" markerWidth="10" markerHeight="10" refX="6" refY="5" orient="auto"><path d="M0 0l8 5-8 5z" fill="#7C93A0"/></marker></defs>
      <path d="M430 66 L520 250" fill="none" stroke="#8FA3AE" stroke-width="2.4" marker-end="url(#ov)"/>
      <path d="M470 300 L290 300" fill="none" stroke="#8FA3AE" stroke-width="2.4" marker-end="url(#ov)"/>
      <path d="M240 250 L330 66" fill="none" stroke="#8FA3AE" stroke-width="2.4" marker-end="url(#ov)"/>
      <rect x="300" y="20" width="160" height="46" rx="23" fill="#EAF2FB" stroke="#4299E0" stroke-width="2.4"/>
      <text x="380" y="49" text-anchor="middle" font-size="18" font-weight="800" fill="#2272B4" font-family="DM Sans,sans-serif">Query</text>
      <rect x="470" y="256" width="160" height="46" rx="23" fill="#EDE7F6" stroke="#7E57C2" stroke-width="2.4"/>
      <text x="550" y="285" text-anchor="middle" font-size="18" font-weight="800" fill="#5E35A1" font-family="DM Sans,sans-serif">Context</text>
      <rect x="130" y="256" width="160" height="46" rx="23" fill="#E8F5E9" stroke="#00A972" stroke-width="2.4"/>
      <text x="210" y="285" text-anchor="middle" font-size="18" font-weight="800" fill="#1B6E48" font-family="DM Sans,sans-serif">Response</text>
      <rect x="300" y="150" width="160" height="40" rx="5" fill="#12313A"/>
      <text x="380" y="176" text-anchor="middle" font-size="16" font-weight="800" fill="#fff" font-family="DM Sans,sans-serif">Ground Truth</text>
      <g font-family="DM Sans,sans-serif" font-size="13.5" fill="#3a4d54">
        <rect x="500" y="86" width="240" height="52" rx="9" fill="#fff" stroke="#EEEDE9"/><text x="516" y="108">Is the context related</text><text x="516" y="126">to the query?</text>
        <rect x="250" y="318" width="260" height="34" rx="9" fill="#fff" stroke="#EEEDE9"/><text x="266" y="340">Is the response supported by the context?</text>
        <rect x="20" y="86" width="240" height="52" rx="9" fill="#fff" stroke="#EEEDE9"/><text x="36" y="108">Is the answer relevant</text><text x="36" y="126">and accurate?</text>
      </g>
    </svg>
  </div>
</div>

### B3. Context Precision

<div class="mt">
  <style>    @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&display=swap');
    .mt{font-family:'DM Sans','Segoe UI',Arial,sans-serif;max-width:1040px;margin:14px auto 6px;color:#1B3139;}
    .mt *{box-sizing:border-box;}
    .mt-tag{display:inline-block;font-size:11.5px;font-weight:800;letter-spacing:.5px;text-transform:uppercase;color:#fff;background:var(--tag);border-radius:7px;padding:4px 11px;margin-bottom:12px;}
    .mt-row{display:flex;flex-wrap:wrap;gap:22px;align-items:center;}
    .mt-col{flex:1 1 300px;}
    .mt-list{list-style:none;margin:0;padding:0;}
    .mt-list li{position:relative;padding-left:24px;font-size:14.5px;line-height:1.5;color:#3a4d54;margin-bottom:11px;}
    .mt-list li b{color:#1B3139;}
    .mt-list li::before{content:'';position:absolute;left:2px;top:7px;width:9px;height:9px;border-radius:2px;transform:rotate(45deg);background:var(--tag);}
    .mt-diag{flex:1 1 300px;background:linear-gradient(180deg,#FBFAF8,#F4F7F9);border:1px solid #EEEDE9;border-radius:14px;box-shadow:0 5px 14px rgba(27,49,57,.07);padding:14px;}
    .mt-diag svg{width:100%;height:auto;max-width:380px;display:block;margin:0 auto;}
    .mt-ex{margin-top:16px;background:#F8F9FC;border:1px solid #E4E9F2;border-left:5px solid #4299E0;border-radius:12px;padding:14px 18px;}
    .mt-ex b.h{display:block;font-size:12px;font-weight:800;letter-spacing:.5px;text-transform:uppercase;color:#2272B4;margin-bottom:8px;}
    .mt-ex ul{list-style:none;margin:0;padding:0;}
    .mt-ex li{font-size:13.5px;line-height:1.55;color:#3a4d54;margin-bottom:7px;}
    .mt-ex .ok{color:#1B6E48;font-weight:700;} .mt-ex .no{color:#B23A2B;font-weight:700;}</style>
  <span class="mt-tag" style="--tag:#4299E0;">Retrieval metric</span>
  <div class="mt-row">
    <div class="mt-col"><ul class="mt-list" style="--tag:#4299E0;"><li>Signal-to-noise ratio for the retrieved context.</li><li>Based on <b>Query</b> and <b>Context(s)</b>.</li><li>Assesses whether relevant chunks rank higher than irrelevant ones.</li></ul></div>
    <div class="mt-diag"><svg viewBox="0 0 500 240" role="img" aria-label="Query, Context and Response triangle with Ground Truth at the center; the metric relates the highlighted nodes"><defs><marker id="tg" markerWidth="10" markerHeight="10" refX="6" refY="5" orient="auto"><path d="M0 0l8 5-8 5z" fill="#7C93A0"/></marker></defs><path d="M300 52 L360 176" fill="none" stroke="#8FA3AE" stroke-width="2.2" marker-end="url(#tg)"/><path d="M330 214 L150 214" fill="none" stroke="#8FA3AE" stroke-width="2.2" marker-end="url(#tg)"/><path d="M120 176 L180 52" fill="none" stroke="#8FA3AE" stroke-width="2.2" marker-end="url(#tg)"/><line x1="250" y1="44" x2="360" y2="196" stroke="#FF5F46" stroke-width="3" stroke-dasharray="7 5" stroke-linecap="round"/><rect x="190" y="120" width="120" height="36" rx="4" fill="#F1F3F5" stroke="#8FA3AE" stroke-width="1.6"/><text x="250" y="143" text-anchor="middle" font-size="14" font-weight="800" fill="#1B3139" font-family="DM Sans,sans-serif">Ground Truth</text><g><rect x="220" y="30" width="120" height="40" rx="20" fill="#FFEDE9" stroke="#FF5F46" stroke-width="2.6"/><text x="280.0" y="56" text-anchor="middle" font-size="16" font-weight="800" fill="#B23A2B" font-family="DM Sans,sans-serif">Query</text></g><g><rect x="60" y="196" width="120" height="40" rx="20" fill="#F1F3F5" stroke="#B7C0C7" stroke-width="1.6"/><text x="120.0" y="222" text-anchor="middle" font-size="16" font-weight="800" fill="#3a4d54" font-family="DM Sans,sans-serif">Response</text></g><g><rect x="360" y="196" width="120" height="40" rx="20" fill="#FFEDE9" stroke="#FF5F46" stroke-width="2.6"/><text x="420.0" y="222" text-anchor="middle" font-size="16" font-weight="800" fill="#B23A2B" font-family="DM Sans,sans-serif">Context</text></g></svg></div>
  </div>
  <div class="mt-ex"><b class="h">Example</b><ul><li><b>Query</b>: What was Einstein’s role in the development of quantum mechanics?</li><li><b>Ground Truth</b>: Einstein contributed to quantum theory, including early skepticism and later contributions.</li><li><span class="ok">High precision</span>: contexts specifically mention his contributions to quantum theory.</li><li><span class="no">Low precision</span>: contexts broadly discuss his life without addressing quantum mechanics.</li></ul></div>
</div>

### B4. Context Relevance

<div class="mt">
  <style>    @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&display=swap');
    .mt{font-family:'DM Sans','Segoe UI',Arial,sans-serif;max-width:1040px;margin:14px auto 6px;color:#1B3139;}
    .mt *{box-sizing:border-box;}
    .mt-tag{display:inline-block;font-size:11.5px;font-weight:800;letter-spacing:.5px;text-transform:uppercase;color:#fff;background:var(--tag);border-radius:7px;padding:4px 11px;margin-bottom:12px;}
    .mt-row{display:flex;flex-wrap:wrap;gap:22px;align-items:center;}
    .mt-col{flex:1 1 300px;}
    .mt-list{list-style:none;margin:0;padding:0;}
    .mt-list li{position:relative;padding-left:24px;font-size:14.5px;line-height:1.5;color:#3a4d54;margin-bottom:11px;}
    .mt-list li b{color:#1B3139;}
    .mt-list li::before{content:'';position:absolute;left:2px;top:7px;width:9px;height:9px;border-radius:2px;transform:rotate(45deg);background:var(--tag);}
    .mt-diag{flex:1 1 300px;background:linear-gradient(180deg,#FBFAF8,#F4F7F9);border:1px solid #EEEDE9;border-radius:14px;box-shadow:0 5px 14px rgba(27,49,57,.07);padding:14px;}
    .mt-diag svg{width:100%;height:auto;max-width:380px;display:block;margin:0 auto;}
    .mt-ex{margin-top:16px;background:#F8F9FC;border:1px solid #E4E9F2;border-left:5px solid #4299E0;border-radius:12px;padding:14px 18px;}
    .mt-ex b.h{display:block;font-size:12px;font-weight:800;letter-spacing:.5px;text-transform:uppercase;color:#2272B4;margin-bottom:8px;}
    .mt-ex ul{list-style:none;margin:0;padding:0;}
    .mt-ex li{font-size:13.5px;line-height:1.55;color:#3a4d54;margin-bottom:7px;}
    .mt-ex .ok{color:#1B6E48;font-weight:700;} .mt-ex .no{color:#B23A2B;font-weight:700;}</style>
  <span class="mt-tag" style="--tag:#4299E0;">Retrieval metric</span>
  <div class="mt-row">
    <div class="mt-col"><ul class="mt-list" style="--tag:#4299E0;"><li>Measures how relevant the retrieved context is to the query.</li><li>Based on the relationship between <b>Query</b> and retrieved <b>Context(s)</b>.</li><li>Focuses on how well the context supports answering the question, not factual accuracy.</li></ul></div>
    <div class="mt-diag"><svg viewBox="0 0 500 240" role="img" aria-label="Query, Context and Response triangle with Ground Truth at the center; the metric relates the highlighted nodes"><defs><marker id="tg" markerWidth="10" markerHeight="10" refX="6" refY="5" orient="auto"><path d="M0 0l8 5-8 5z" fill="#7C93A0"/></marker></defs><path d="M300 52 L360 176" fill="none" stroke="#8FA3AE" stroke-width="2.2" marker-end="url(#tg)"/><path d="M330 214 L150 214" fill="none" stroke="#8FA3AE" stroke-width="2.2" marker-end="url(#tg)"/><path d="M120 176 L180 52" fill="none" stroke="#8FA3AE" stroke-width="2.2" marker-end="url(#tg)"/><line x1="250" y1="44" x2="360" y2="196" stroke="#FF5F46" stroke-width="3" stroke-dasharray="7 5" stroke-linecap="round"/><rect x="190" y="120" width="120" height="36" rx="4" fill="#F1F3F5" stroke="#8FA3AE" stroke-width="1.6"/><text x="250" y="143" text-anchor="middle" font-size="14" font-weight="800" fill="#1B3139" font-family="DM Sans,sans-serif">Ground Truth</text><g><rect x="220" y="30" width="120" height="40" rx="20" fill="#FFEDE9" stroke="#FF5F46" stroke-width="2.6"/><text x="280.0" y="56" text-anchor="middle" font-size="16" font-weight="800" fill="#B23A2B" font-family="DM Sans,sans-serif">Query</text></g><g><rect x="60" y="196" width="120" height="40" rx="20" fill="#F1F3F5" stroke="#B7C0C7" stroke-width="1.6"/><text x="120.0" y="222" text-anchor="middle" font-size="16" font-weight="800" fill="#3a4d54" font-family="DM Sans,sans-serif">Response</text></g><g><rect x="360" y="196" width="120" height="40" rx="20" fill="#FFEDE9" stroke="#FF5F46" stroke-width="2.6"/><text x="420.0" y="222" text-anchor="middle" font-size="16" font-weight="800" fill="#B23A2B" font-family="DM Sans,sans-serif">Context</text></g></svg></div>
  </div>
  <div class="mt-ex"><b class="h">Example</b><ul><li><b>Query</b>: What was Einstein’s role in the development of quantum mechanics?</li><li><span class="ok">High relevance</span>: Einstein initially challenged quantum theory but later contributed foundational ideas.</li><li><span class="no">Low relevance</span>: Einstein held pacifist views and became a U.S. citizen in 1940.</li></ul></div>
</div>

### B5. Context Recall

<div class="mt">
  <style>    @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&display=swap');
    .mt{font-family:'DM Sans','Segoe UI',Arial,sans-serif;max-width:1040px;margin:14px auto 6px;color:#1B3139;}
    .mt *{box-sizing:border-box;}
    .mt-tag{display:inline-block;font-size:11.5px;font-weight:800;letter-spacing:.5px;text-transform:uppercase;color:#fff;background:var(--tag);border-radius:7px;padding:4px 11px;margin-bottom:12px;}
    .mt-row{display:flex;flex-wrap:wrap;gap:22px;align-items:center;}
    .mt-col{flex:1 1 300px;}
    .mt-list{list-style:none;margin:0;padding:0;}
    .mt-list li{position:relative;padding-left:24px;font-size:14.5px;line-height:1.5;color:#3a4d54;margin-bottom:11px;}
    .mt-list li b{color:#1B3139;}
    .mt-list li::before{content:'';position:absolute;left:2px;top:7px;width:9px;height:9px;border-radius:2px;transform:rotate(45deg);background:var(--tag);}
    .mt-diag{flex:1 1 300px;background:linear-gradient(180deg,#FBFAF8,#F4F7F9);border:1px solid #EEEDE9;border-radius:14px;box-shadow:0 5px 14px rgba(27,49,57,.07);padding:14px;}
    .mt-diag svg{width:100%;height:auto;max-width:380px;display:block;margin:0 auto;}
    .mt-ex{margin-top:16px;background:#F8F9FC;border:1px solid #E4E9F2;border-left:5px solid #4299E0;border-radius:12px;padding:14px 18px;}
    .mt-ex b.h{display:block;font-size:12px;font-weight:800;letter-spacing:.5px;text-transform:uppercase;color:#2272B4;margin-bottom:8px;}
    .mt-ex ul{list-style:none;margin:0;padding:0;}
    .mt-ex li{font-size:13.5px;line-height:1.55;color:#3a4d54;margin-bottom:7px;}
    .mt-ex .ok{color:#1B6E48;font-weight:700;} .mt-ex .no{color:#B23A2B;font-weight:700;}</style>
  <span class="mt-tag" style="--tag:#4299E0;">Retrieval metric</span>
  <div class="mt-row">
    <div class="mt-col"><ul class="mt-list" style="--tag:#4299E0;"><li>Measures the extent to which all relevant information is retrieved and present in the context.</li><li>Based on <b>Ground Truth</b> and <b>Context(s)</b>.</li></ul></div>
    <div class="mt-diag"><svg viewBox="0 0 500 240" role="img" aria-label="Query, Context and Response triangle with Ground Truth at the center; the metric relates the highlighted nodes"><defs><marker id="tg" markerWidth="10" markerHeight="10" refX="6" refY="5" orient="auto"><path d="M0 0l8 5-8 5z" fill="#7C93A0"/></marker></defs><path d="M300 52 L360 176" fill="none" stroke="#8FA3AE" stroke-width="2.2" marker-end="url(#tg)"/><path d="M330 214 L150 214" fill="none" stroke="#8FA3AE" stroke-width="2.2" marker-end="url(#tg)"/><path d="M120 176 L180 52" fill="none" stroke="#8FA3AE" stroke-width="2.2" marker-end="url(#tg)"/><line x1="360" y1="196" x2="250" y2="138" stroke="#FF5F46" stroke-width="3" stroke-dasharray="7 5" stroke-linecap="round"/><rect x="190" y="120" width="120" height="36" rx="4" fill="#FFEDE9" stroke="#FF5F46" stroke-width="2.6"/><text x="250" y="143" text-anchor="middle" font-size="14" font-weight="800" fill="#B23A2B" font-family="DM Sans,sans-serif">Ground Truth</text><g><rect x="220" y="30" width="120" height="40" rx="20" fill="#F1F3F5" stroke="#B7C0C7" stroke-width="1.6"/><text x="280.0" y="56" text-anchor="middle" font-size="16" font-weight="800" fill="#3a4d54" font-family="DM Sans,sans-serif">Query</text></g><g><rect x="60" y="196" width="120" height="40" rx="20" fill="#F1F3F5" stroke="#B7C0C7" stroke-width="1.6"/><text x="120.0" y="222" text-anchor="middle" font-size="16" font-weight="800" fill="#3a4d54" font-family="DM Sans,sans-serif">Response</text></g><g><rect x="360" y="196" width="120" height="40" rx="20" fill="#FFEDE9" stroke="#FF5F46" stroke-width="2.6"/><text x="420.0" y="222" text-anchor="middle" font-size="16" font-weight="800" fill="#B23A2B" font-family="DM Sans,sans-serif">Context</text></g></svg></div>
  </div>
  <div class="mt-ex"><b class="h">Example</b><ul><li><b>Query</b>: What significant scientific theories did Einstein contribute to?</li><li><b>Ground Truth</b>: relativity and insights into quantum mechanics.</li><li><span class="ok">High-recall context</span>: Einstein contributed to relativity and quantum mechanics.</li><li><span class="no">Low-recall context</span>: Einstein contributed to relativity.</li></ul></div>
</div>

### B6. Faithfulness

<div class="mt">
  <style>    @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&display=swap');
    .mt{font-family:'DM Sans','Segoe UI',Arial,sans-serif;max-width:1040px;margin:14px auto 6px;color:#1B3139;}
    .mt *{box-sizing:border-box;}
    .mt-tag{display:inline-block;font-size:11.5px;font-weight:800;letter-spacing:.5px;text-transform:uppercase;color:#fff;background:var(--tag);border-radius:7px;padding:4px 11px;margin-bottom:12px;}
    .mt-row{display:flex;flex-wrap:wrap;gap:22px;align-items:center;}
    .mt-col{flex:1 1 300px;}
    .mt-list{list-style:none;margin:0;padding:0;}
    .mt-list li{position:relative;padding-left:24px;font-size:14.5px;line-height:1.5;color:#3a4d54;margin-bottom:11px;}
    .mt-list li b{color:#1B3139;}
    .mt-list li::before{content:'';position:absolute;left:2px;top:7px;width:9px;height:9px;border-radius:2px;transform:rotate(45deg);background:var(--tag);}
    .mt-diag{flex:1 1 300px;background:linear-gradient(180deg,#FBFAF8,#F4F7F9);border:1px solid #EEEDE9;border-radius:14px;box-shadow:0 5px 14px rgba(27,49,57,.07);padding:14px;}
    .mt-diag svg{width:100%;height:auto;max-width:380px;display:block;margin:0 auto;}
    .mt-ex{margin-top:16px;background:#F8F9FC;border:1px solid #E4E9F2;border-left:5px solid #4299E0;border-radius:12px;padding:14px 18px;}
    .mt-ex b.h{display:block;font-size:12px;font-weight:800;letter-spacing:.5px;text-transform:uppercase;color:#2272B4;margin-bottom:8px;}
    .mt-ex ul{list-style:none;margin:0;padding:0;}
    .mt-ex li{font-size:13.5px;line-height:1.55;color:#3a4d54;margin-bottom:7px;}
    .mt-ex .ok{color:#1B6E48;font-weight:700;} .mt-ex .no{color:#B23A2B;font-weight:700;}</style>
  <span class="mt-tag" style="--tag:#00A972;">Generation metric</span>
  <div class="mt-row">
    <div class="mt-col"><ul class="mt-list" style="--tag:#00A972;"><li>Measures the factual accuracy of the generated answer relative to the provided context.</li><li>Based on the <b>Response</b> and <b>retrieved Context(s)</b>.</li></ul></div>
    <div class="mt-diag"><svg viewBox="0 0 500 240" role="img" aria-label="Query, Context and Response triangle with Ground Truth at the center; the metric relates the highlighted nodes"><defs><marker id="tg" markerWidth="10" markerHeight="10" refX="6" refY="5" orient="auto"><path d="M0 0l8 5-8 5z" fill="#7C93A0"/></marker></defs><path d="M300 52 L360 176" fill="none" stroke="#8FA3AE" stroke-width="2.2" marker-end="url(#tg)"/><path d="M330 214 L150 214" fill="none" stroke="#8FA3AE" stroke-width="2.2" marker-end="url(#tg)"/><path d="M120 176 L180 52" fill="none" stroke="#8FA3AE" stroke-width="2.2" marker-end="url(#tg)"/><line x1="360" y1="196" x2="160" y2="196" stroke="#FF5F46" stroke-width="3" stroke-dasharray="7 5" stroke-linecap="round"/><rect x="190" y="120" width="120" height="36" rx="4" fill="#F1F3F5" stroke="#8FA3AE" stroke-width="1.6"/><text x="250" y="143" text-anchor="middle" font-size="14" font-weight="800" fill="#1B3139" font-family="DM Sans,sans-serif">Ground Truth</text><g><rect x="220" y="30" width="120" height="40" rx="20" fill="#F1F3F5" stroke="#B7C0C7" stroke-width="1.6"/><text x="280.0" y="56" text-anchor="middle" font-size="16" font-weight="800" fill="#3a4d54" font-family="DM Sans,sans-serif">Query</text></g><g><rect x="60" y="196" width="120" height="40" rx="20" fill="#FFEDE9" stroke="#FF5F46" stroke-width="2.6"/><text x="120.0" y="222" text-anchor="middle" font-size="16" font-weight="800" fill="#B23A2B" font-family="DM Sans,sans-serif">Response</text></g><g><rect x="360" y="196" width="120" height="40" rx="20" fill="#FFEDE9" stroke="#FF5F46" stroke-width="2.6"/><text x="420.0" y="222" text-anchor="middle" font-size="16" font-weight="800" fill="#B23A2B" font-family="DM Sans,sans-serif">Context</text></g></svg></div>
  </div>
  <div class="mt-ex"><b class="h">Example</b><ul><li><b>Query</b>: Where and when was Einstein born?</li><li><b>Context</b>: Albert Einstein (born 14 March 1879) was a German-born theoretical physicist.</li><li><span class="ok">High faithfulness</span>: Einstein was born in Germany on 14 March 1879.</li><li><span class="no">Low faithfulness</span>: Einstein was born in Germany on 20 March 1879.</li></ul></div>
</div>

### B7. Answer Relevancy

<div class="mt">
  <style>    @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&display=swap');
    .mt{font-family:'DM Sans','Segoe UI',Arial,sans-serif;max-width:1040px;margin:14px auto 6px;color:#1B3139;}
    .mt *{box-sizing:border-box;}
    .mt-tag{display:inline-block;font-size:11.5px;font-weight:800;letter-spacing:.5px;text-transform:uppercase;color:#fff;background:var(--tag);border-radius:7px;padding:4px 11px;margin-bottom:12px;}
    .mt-row{display:flex;flex-wrap:wrap;gap:22px;align-items:center;}
    .mt-col{flex:1 1 300px;}
    .mt-list{list-style:none;margin:0;padding:0;}
    .mt-list li{position:relative;padding-left:24px;font-size:14.5px;line-height:1.5;color:#3a4d54;margin-bottom:11px;}
    .mt-list li b{color:#1B3139;}
    .mt-list li::before{content:'';position:absolute;left:2px;top:7px;width:9px;height:9px;border-radius:2px;transform:rotate(45deg);background:var(--tag);}
    .mt-diag{flex:1 1 300px;background:linear-gradient(180deg,#FBFAF8,#F4F7F9);border:1px solid #EEEDE9;border-radius:14px;box-shadow:0 5px 14px rgba(27,49,57,.07);padding:14px;}
    .mt-diag svg{width:100%;height:auto;max-width:380px;display:block;margin:0 auto;}
    .mt-ex{margin-top:16px;background:#F8F9FC;border:1px solid #E4E9F2;border-left:5px solid #4299E0;border-radius:12px;padding:14px 18px;}
    .mt-ex b.h{display:block;font-size:12px;font-weight:800;letter-spacing:.5px;text-transform:uppercase;color:#2272B4;margin-bottom:8px;}
    .mt-ex ul{list-style:none;margin:0;padding:0;}
    .mt-ex li{font-size:13.5px;line-height:1.55;color:#3a4d54;margin-bottom:7px;}
    .mt-ex .ok{color:#1B6E48;font-weight:700;} .mt-ex .no{color:#B23A2B;font-weight:700;}</style>
  <span class="mt-tag" style="--tag:#00A972;">Generation metric</span>
  <div class="mt-row">
    <div class="mt-col"><ul class="mt-list" style="--tag:#00A972;"><li>Assesses how pertinent and applicable the response is to the user’s query.</li><li>Based on the alignment of the <b>Response</b> with the <b>Query</b>.</li></ul></div>
    <div class="mt-diag"><svg viewBox="0 0 500 240" role="img" aria-label="Query, Context and Response triangle with Ground Truth at the center; the metric relates the highlighted nodes"><defs><marker id="tg" markerWidth="10" markerHeight="10" refX="6" refY="5" orient="auto"><path d="M0 0l8 5-8 5z" fill="#7C93A0"/></marker></defs><path d="M300 52 L360 176" fill="none" stroke="#8FA3AE" stroke-width="2.2" marker-end="url(#tg)"/><path d="M330 214 L150 214" fill="none" stroke="#8FA3AE" stroke-width="2.2" marker-end="url(#tg)"/><path d="M120 176 L180 52" fill="none" stroke="#8FA3AE" stroke-width="2.2" marker-end="url(#tg)"/><line x1="250" y1="44" x2="160" y2="196" stroke="#FF5F46" stroke-width="3" stroke-dasharray="7 5" stroke-linecap="round"/><rect x="190" y="120" width="120" height="36" rx="4" fill="#F1F3F5" stroke="#8FA3AE" stroke-width="1.6"/><text x="250" y="143" text-anchor="middle" font-size="14" font-weight="800" fill="#1B3139" font-family="DM Sans,sans-serif">Ground Truth</text><g><rect x="220" y="30" width="120" height="40" rx="20" fill="#FFEDE9" stroke="#FF5F46" stroke-width="2.6"/><text x="280.0" y="56" text-anchor="middle" font-size="16" font-weight="800" fill="#B23A2B" font-family="DM Sans,sans-serif">Query</text></g><g><rect x="60" y="196" width="120" height="40" rx="20" fill="#FFEDE9" stroke="#FF5F46" stroke-width="2.6"/><text x="120.0" y="222" text-anchor="middle" font-size="16" font-weight="800" fill="#B23A2B" font-family="DM Sans,sans-serif">Response</text></g><g><rect x="360" y="196" width="120" height="40" rx="20" fill="#F1F3F5" stroke="#B7C0C7" stroke-width="1.6"/><text x="420.0" y="222" text-anchor="middle" font-size="16" font-weight="800" fill="#3a4d54" font-family="DM Sans,sans-serif">Context</text></g></svg></div>
  </div>
  <div class="mt-ex"><b class="h">Example</b><ul><li><b>Query</b>: What is Einstein known for?</li><li><span class="ok">High relevancy</span>: Einstein is known for developing the theory of relativity.</li><li><span class="no">Low relevancy</span>: Einstein was a scientist.</li></ul></div>
</div>

### B8. Answer Correctness

<div class="mt">
  <style>    @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&display=swap');
    .mt{font-family:'DM Sans','Segoe UI',Arial,sans-serif;max-width:1040px;margin:14px auto 6px;color:#1B3139;}
    .mt *{box-sizing:border-box;}
    .mt-tag{display:inline-block;font-size:11.5px;font-weight:800;letter-spacing:.5px;text-transform:uppercase;color:#fff;background:var(--tag);border-radius:7px;padding:4px 11px;margin-bottom:12px;}
    .mt-row{display:flex;flex-wrap:wrap;gap:22px;align-items:center;}
    .mt-col{flex:1 1 300px;}
    .mt-list{list-style:none;margin:0;padding:0;}
    .mt-list li{position:relative;padding-left:24px;font-size:14.5px;line-height:1.5;color:#3a4d54;margin-bottom:11px;}
    .mt-list li b{color:#1B3139;}
    .mt-list li::before{content:'';position:absolute;left:2px;top:7px;width:9px;height:9px;border-radius:2px;transform:rotate(45deg);background:var(--tag);}
    .mt-diag{flex:1 1 300px;background:linear-gradient(180deg,#FBFAF8,#F4F7F9);border:1px solid #EEEDE9;border-radius:14px;box-shadow:0 5px 14px rgba(27,49,57,.07);padding:14px;}
    .mt-diag svg{width:100%;height:auto;max-width:380px;display:block;margin:0 auto;}
    .mt-ex{margin-top:16px;background:#F8F9FC;border:1px solid #E4E9F2;border-left:5px solid #4299E0;border-radius:12px;padding:14px 18px;}
    .mt-ex b.h{display:block;font-size:12px;font-weight:800;letter-spacing:.5px;text-transform:uppercase;color:#2272B4;margin-bottom:8px;}
    .mt-ex ul{list-style:none;margin:0;padding:0;}
    .mt-ex li{font-size:13.5px;line-height:1.55;color:#3a4d54;margin-bottom:7px;}
    .mt-ex .ok{color:#1B6E48;font-weight:700;} .mt-ex .no{color:#B23A2B;font-weight:700;}</style>
  <span class="mt-tag" style="--tag:#00A972;">Generation metric</span>
  <div class="mt-row">
    <div class="mt-col"><ul class="mt-list" style="--tag:#00A972;"><li>Measures the accuracy of the generated answer compared to the ground truth.</li><li>Based on the <b>Ground Truth</b> and the <b>Response</b>.</li><li>Encompasses both semantic and factual similarity.</li></ul></div>
    <div class="mt-diag"><svg viewBox="0 0 500 240" role="img" aria-label="Query, Context and Response triangle with Ground Truth at the center; the metric relates the highlighted nodes"><defs><marker id="tg" markerWidth="10" markerHeight="10" refX="6" refY="5" orient="auto"><path d="M0 0l8 5-8 5z" fill="#7C93A0"/></marker></defs><path d="M300 52 L360 176" fill="none" stroke="#8FA3AE" stroke-width="2.2" marker-end="url(#tg)"/><path d="M330 214 L150 214" fill="none" stroke="#8FA3AE" stroke-width="2.2" marker-end="url(#tg)"/><path d="M120 176 L180 52" fill="none" stroke="#8FA3AE" stroke-width="2.2" marker-end="url(#tg)"/><line x1="160" y1="196" x2="250" y2="138" stroke="#FF5F46" stroke-width="3" stroke-dasharray="7 5" stroke-linecap="round"/><rect x="190" y="120" width="120" height="36" rx="4" fill="#FFEDE9" stroke="#FF5F46" stroke-width="2.6"/><text x="250" y="143" text-anchor="middle" font-size="14" font-weight="800" fill="#B23A2B" font-family="DM Sans,sans-serif">Ground Truth</text><g><rect x="220" y="30" width="120" height="40" rx="20" fill="#F1F3F5" stroke="#B7C0C7" stroke-width="1.6"/><text x="280.0" y="56" text-anchor="middle" font-size="16" font-weight="800" fill="#3a4d54" font-family="DM Sans,sans-serif">Query</text></g><g><rect x="60" y="196" width="120" height="40" rx="20" fill="#FFEDE9" stroke="#FF5F46" stroke-width="2.6"/><text x="120.0" y="222" text-anchor="middle" font-size="16" font-weight="800" fill="#B23A2B" font-family="DM Sans,sans-serif">Response</text></g><g><rect x="360" y="196" width="120" height="40" rx="20" fill="#F1F3F5" stroke="#B7C0C7" stroke-width="1.6"/><text x="420.0" y="222" text-anchor="middle" font-size="16" font-weight="800" fill="#3a4d54" font-family="DM Sans,sans-serif">Context</text></g></svg></div>
  </div>
  <div class="mt-ex"><b class="h">Example</b><ul><li><b>Ground Truth</b>: Einstein won the Nobel Prize in Physics in 1921 for the photoelectric effect.</li><li><span class="ok">High correctness</span>: Einstein received the Nobel Prize in Physics in 1921 for the photoelectric effect.</li><li><span class="no">Low correctness</span>: Einstein won the Nobel Prize in the 1930s for relativity.</li></ul></div>
</div>


## C. Custom Metrics


### C1. Custom Metrics for System Evaluation

#### Frequently related to business goals and constraints for the AI system

- What is important to your use case?
    - Do you care about serving latency?
    - Are you concerned with total cost?
    - Are you expecting your system to increase product demand?
    - What about customer satisfaction?

**Note:** custom metrics can be useful for individual components, too.

<br>
<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Quick Activity</strong>
<p style="margin: 8px 0 0 0; color: #333;">Define your own system-wide custom metric to ensure your AI system is providing the value that you expect.</p>
</div>
</div>
</div>


### C2. Custom Metrics in MLflow

#### Built-in capabilities for integrating custom metrics into component monitoring

<table style="width: 100% !important; max-width: 100% !important; border-collapse: collapse; table-layout: fixed; margin: 0; padding: 0;">
  <colgroup>
    <col style="width: 40%;">
    <col style="width: 60%;">
  </colgroup>
  <tr>
    <td style="vertical-align: top; padding-right: 20px; box-sizing: border-box;">
      <br>
      <ul style="font-size: 2em;">
        <li><b>Custom Guidelines</b>: Define evaluation rules in natural language for flexible scoring.</li>
        <li><b>Unified Scoring</b>: Combine custom and built-in evaluators in one run.</li>
        <li><b>Tracked Results</b>: Auto-log scores and aggregates in MLflow experiments.</li>
      </ul>
    </td>
    <td style="vertical-align: top; padding-left: 20px; box-sizing: border-box;">
      <div class="code-block-dark" data-language="sql" data-source="raw-sql-1"></div>
    </td>
  </tr>
</table>

<textarea id="raw-sql-1" style="display:none;">
from mlflow.metrics.genai import evaluate, Guidelines, RelevanceToQuery, Safety
import mlflow

# (1) Define the professionalism rubric as a Guidelines scorer
professionalism_guidelines = Guidelines(
    name="professionalism",
    guidelines="""Assess the tone of the model's answer for professionalism.

Score using this scale (integer only):
- 0: Extremely casual/informal; slang or colloquialisms; not suitable for professional contexts.
- 1: Casual but respectful; avoids strong informality; acceptable only in informal professional settings.
- 2: Mostly formal but with some casual words/phrases; borderline for professional contexts.
- 3: Balanced, clear, and neutral; suitable for most professional contexts.

Instruction:
- Judge ONLY tone/style, not factual accuracy or relevance.
- Provide a brief justification.
- Return a numeric score (0–3). For pass/fail aggregation, treat scores ≥2 as PASS.
"""
)

# (2) Run evaluation (eval_set has "inputs" and optional "expectations")

with mlflow.start_run(run_name="eval_professionalism") as eval_run:
    eval_results = evaluate(
        data=DA.eval_set,
        predict_fn=predict_fn,
        scorers=[
            professionalism_guidelines,  # replaces the old custom metric
            RelevanceToQuery(),          # optional, built-in
            Safety(),                    # optional, built-in
        ],
    )
</textarea>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism-tomorrow.min.css" rel="stylesheet" id="prism-dark-theme" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>


<script>
(function() {
    document.querySelectorAll('.code-block-dark').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');

        var lang = block.getAttribute('data-language') || 'sql';
        var sourceId = block.getAttribute('data-source');

        var raw;
        if (sourceId) {
            var rawEl = document.getElementById(sourceId);
            if (!rawEl) return;
            raw = rawEl.value;
        } else {
            raw = block.textContent;
        }

        var code = raw.trim();
        var id = 'code-dark-' + Math.random().toString(36).substr(2, 9);

        block.innerHTML =
            '<div style="position:relative;margin:16px 0;max-width:100%;">' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#555;color:#fff;border:1px solid #666;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#2d2d2d;border-radius:8px;padding:16px;padding-top:40px;overflow-x:auto;margin:0;border:1px solid #444;max-width:100%;box-sizing:border-box;">' +
                    '<code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;word-wrap:break-word;white-space:pre-wrap;"></code>' +
                '</pre>' +
            '</div>';

        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);

        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>


## D. Human Feedback and Monitoring


### D1. Human Feedback

#### Collect data from users and experts

- Often developers are not the experts of the domain

- Models’ output need to be **evaluated by human experts**

- Models’ outputs and associated feedback need to be collected and stored in a structured manner

- Feedback can be **explicit or implicit**:

    - **Explicit feedback**: Direct and intentional input from users, such as ratings, comments and review.

    - **Implicit feedback**: Gathered indirectly by observing user behavior and interactions, such as engagement metrics and behavioral data.

### D2. Collect Human Feedback with Review App

<div class="rv">
  <style>
    @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&display=swap');
    .rv{font-family:'DM Sans','Segoe UI',Arial,sans-serif;max-width:1040px;margin:14px auto 6px;color:#1B3139;}
    .rv .sub{font-size:14px;color:#5A6F77;margin:0 0 14px;}
    .rv-frame{background:#fff;border:1px solid #EEEDE9;border-radius:14px;padding:10px;box-shadow:0 8px 20px rgba(27,49,57,.12);}
    .rv-frame img{display:block;width:100%;height:auto;border-radius:8px;}
  </style>
  <p class="sub">Agent Bricks custom agents provide a pre-built review app so domain experts can rate responses and leave feedback.</p>
  <div class="rv-frame"><img src="../Includes/images/M4L1_77.png" alt="Screenshot of the MLflow review app where experts review chats and answer questions"></div>
</div>


### D3. Monitoring Quality in Production

#### Automatically run quality checks for AI systems in production

- Automatically run MLflow scorers on traces from your production GenAI apps to **continuously monitor quality**.

- Use the **same scorers in development and production** to ensure consistent evaluation.

- Continuous quality assessment with monitoring **running in the background**.

## E. Hands-on: Evaluate the RAG Model with MLflow

Now that we know *what* to measure and *why*, let's evaluate the RAG model we registered in the previous lesson. In the sections that follow, we will run the classroom setup, load the model from the registry, define an evaluation dataset, run built-in and custom metrics, and review the results.


### E1. Classroom Setup

Set up the classroom environment by installing the required libraries and datasets.

In [0]:
%run ../Includes/Classroom-Setup-03

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


/home/spark-cc925dd1-d34f-4ddc-abec-ee/.ipykernel/308/command-5842534141863397-689038187:9: DeprecationWarning: databricks-vectorsearch is deprecated and has been renamed to databricks-ai-search. Imports under 'databricks.vector_search.*' will continue to work as a thin re-export of 'databricks.ai_search.*', but new code should switch to 'pip install databricks-ai-search' and 'from databricks.ai_search.* import ...'.
  from databricks.vector_search.client import VectorSearchClient



The examples and models presented in this course are intended solely for demonstration and educational purposes.
 Please note that the models and prompt examples may sometimes contain offensive, inaccurate, biased, or harmful content.


### E2. Prepare for Evaluation

The first step is to load the model that will be used for evaluation. The RAG model we built in the previous demo was **registered in the UC model registry**.

After loading the model, we will **define the evaluation dataset**. This dataset must include a **"request"** field to be used as the input query for the model. It typically also includes **"ground-truth" fields for response or context**. In this demo, to keep it simple, we will use only one request-response pair.



#### Load the Model

Let's start by loading the model so that we can use it to generate responses and evaluate those responses later. Note that we will load a specific version of the model.

In [0]:
import mlflow
mlflow.set_registry_uri("databricks-uc")

model_uri = f"models:/{DA.catalog_name}.{DA.schema_name}.getstarted_genai_retrevial_demo/1"
print(model_uri)

models:/learn_get_started_genai_puli_reddy.default.getstarted_genai_retrevial_demo/1


#### Load the Evaluation Dataset

In this demo we will only use **"request" and "expected_response"**. For examples of using other fields, you can check [sample evaluation sets](https://docs.databricks.com/en/generative-ai/agent-evaluation/evaluation-set.html#sample-evaluations-sets). 

Evaluation dataset is crucial for the accuracy of the evaluation. Some of the best practices are; 
* When developing an evaluation set, treat each sample as a unit test with a specific scenario and expected outcome, **including challenging examples and adversarial scenarios**. 

* Ensure the evaluation set **reflects a variety of questions**, even beyond the application's primary domain, to prevent hallucinations or harmful responses. 

* **High-quality, consistent human-generated labels are crucial**; achieve this by aggregating responses from multiple labelers and providing clear instructions to ensure consistency.

In [0]:
display(DA.eval_set)

inputs,expectations
"List(List(List(personalized wooden name sign with app control, user)))",List(true)
"List(List(List(handmade lamp with remote dimming, user)))",List(true)
"List(List(List(ceramic planter with self-watering insert and moisture indicator, user)))",List(false)


The next cell prepares everything needed for the first evaluation run:

1. Sets the AI Search index name as an environment variable used by the agent.
2. Loads the registered model from the Unity Catalog Model Registry.
3. Defines a `predict_fn` function that calls the model and extracts the assistant's text response for the scorers.
4. Defines a `Guidelines` scorer (`creative_enough`) in natural language.
5. Runs `mlflow.genai.evaluate` with the custom scorer plus the built-in `RelevanceToQuery` and `Safety` scorers, logging the run to MLflow.

In [0]:
import os
import mlflow
from mlflow.genai import evaluate
from mlflow.genai.scorers import Guidelines, RelevanceToQuery, Safety
import mlflow.pyfunc

os.environ["VS_INDEX_NAME"] = f"{DA.catalog_name}.{DA.schema_name}.product_embeddings"

# (1) Predict fn that calls your registered model

model = mlflow.pyfunc.load_model(model_uri)

# Predict function expected by mlflow.genai.evaluate()
@mlflow.trace
def predict_fn(messages):
    """
    messages: list of {role, content}
    Returns: assistant text (str) for scorers.
    """
    out = model.predict({"input": messages})

    # Databricks Responses shape: {"object":"response","output":[{message...}]}
    if isinstance(out, dict) and "output" in out:
        try:
            return out["output"][-1]["content"][0]["text"].strip()
        except Exception:
            pass  # fall through

    # Older shape: list of message dicts
    if isinstance(out, list):
        try:
            return out[-1]["content"][0]["text"].strip()
        except Exception:
            pass

    # Fallbacks
    if isinstance(out, str):
        return out.strip()
    if isinstance(out, dict):
        for k in ("answer", "response", "text", "output"):
            v = out.get(k)
            if isinstance(v, str):
                return v.strip()

    return str(out).strip()

# Guidelines scorer for creative_enough (boolean pass/fail)
creative_guidelines = Guidelines(
    name="creative_enough",
    guidelines="""The response must feel original and imaginative, not a paraphrase of the prompt.
It should introduce a concept or angle (e.g., theme, use-case, material pairing) beyond listing features.
Keep within ~250 words and avoid tool/planning narration."""
)

# (3) Run evaluation (eval_set has "inputs" and optional "expectations")
mlflow.set_experiment(f"/Users/{DA.username}/Retreival_Eval")
with mlflow.start_run(run_name="eval_guidelines") as eval_run:
    eval_results = evaluate(
        data=DA.eval_set, 
        predict_fn=predict_fn, 
        scorers=[
            creative_guidelines,     # pass/fail vs. our creative_enough label
            RelevanceToQuery(),      # built-in; no ground truth required
            Safety(),                # built-in; no ground truth required
        ]
    )

Unexpected status resolving file endpoints (403, request ID 38f8e0ed-dc6a-42f2-8fb9-c00f8c51b962), using the default endpoints.


2026/11/16 18:06:53 INFO mlflow.tracking.fluent: Experiment with name '/Users/puli.reddy@gmail.com/Retreival_Eval' does not exist. Creating a new experiment.
If you are using MLflow Tracing, consider storing your traces in Unity Catalog for unlimited storage (no 100,000 trace limit), fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog
2026/11/16 18:06:56 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/11/16 18:06:56 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/3 [Elapsed: 00:00, Remaining: ?]

### E3. Model Evaluation

#### Built-in Metrics

The Agent Bricks Custom Agents uses LLM-judges to evaluate the model using common evaluation metrics such as safety and relevance. Also, it calculates system metrics such as token count and latency.    

#### Custom Metric

Although the Agents Evaluation framework automatically calculates common evaluation metrics, there are instances where we may need to assess the model using custom metrics. In this section, we will define a custom metric to evaluate whether the **response** generated is creative or not.

In [0]:
from mlflow.genai.scorers import Guidelines

creativity_level = Guidelines(
    name="creativity_level",
    guidelines="""Your task is to determine whether the generated product idea is creative or not.
Grade the idea from 1 to 5, where 5 is very creative and 1 is very boring.
This is the output: {response}."""
)

#### Run Evaluation Test

Please note that in the code below, we are logging the evaluation process using MLflow to enable viewing the results through the MLflow UI.

In [0]:
DA.eval_set["inputs"] = DA.eval_set["inputs"].apply(
    lambda x: {
        "input": [
            {
                "role": "user",
                "content": x["messages"][0]["content"]
            }
        ]
    }
)

display(DA.eval_set)

inputs,expectations
"List(List(List(personalized wooden name sign with app control, user)))",List(true)
"List(List(List(handmade lamp with remote dimming, user)))",List(true)
"List(List(List(ceramic planter with self-watering insert and moisture indicator, user)))",List(false)


Now run the evaluation with `mlflow.evaluate`, using the `databricks-agent` model type and passing the custom `creativity_level` metric alongside the built-in metrics. When the run completes, an **Experiment** link appears in the output — you will use it in the next section.

In [0]:
print(model_uri)
with mlflow.start_run(run_name="rag_eval_demo_04_01"):
    eval_results = mlflow.evaluate(
        data=DA.eval_set,
        model = model_uri,
        model_type = "databricks-agent",
        extra_metrics=[creativity_level]
    )

models:/learn_get_started_genai_puli_reddy.default.getstarted_genai_retrevial_demo/1


/local_disk0/.ephemeral_nfs/envs/pythonEnv-cc925dd1-d34f-4ddc-abec-ee9bf11a2293/lib/python3.12/site-packages/mlflow/models/evaluation/deprecated.py:32: FutureWarning: The `mlflow.evaluate` API has been deprecated as of MLflow 3.0.0. Please use these new alternatives:

 - For traditional ML or deep learning models: Use `mlflow.models.evaluate`, which maintains full compatibility with the original `mlflow.evaluate` API.

 - For LLMs or GenAI applications: Use the new `mlflow.genai.evaluate` API, which offers enhanced features specifically designed for evaluating LLMs and GenAI applications.

  warnings.warn(warning_message, FutureWarning)


2026/11/16 18:09:09 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-3105be65461a4f77a5a3a46223af2b6b
2026/11/16 18:09:09 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.


Evaluating:   0%|          | 0/3 [Elapsed: 00:00, Remaining: ?] 

/opt/databricks-environments/databricks-ml/lib/python3.12/site-packages/databricks/rag_eval/mlflow/databricks_rag_evaluator.py:158: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  search_traces_df = mlflow.search_traces(experiment_ids=[experiment_id], run_id=run_id)


<!DOCTYPE html>
 
 
 Evaluation output 
 
 
 
 
 
 
 
 
 View evaluation results.

[Trace(trace_id='tr-9a4d3459f056203eb9652e81cae55629'), Trace(trace_id='tr-c0c48614cb97fc05cd6b318474f611ba'), Trace(trace_id='tr-8d73fc3e3b23a30b43b9f1fbb4d94553')]

### E4. Review Evaluation Results

We have two options for reviewing the evaluation results. The first option is to examine the metrics and tables directly using the results object. The second option is to review the results through the user interface (UI). 

#### Review Results via the UI

To view the results in the UI, follow these steps:

1. Click the **Experiment** link displayed at the top of the previous code block's output. Alternatively, navigate to **Experiments** in the left panel and locate the experiment registered with the title of this notebook.
2. View the overall metrics in the **Model Metrics** tab.
3. Examine detailed results for each assessment in the **Evaluation Results** tab.

#### Review the Results Table

In [0]:
display(eval_results.metrics)

{'agent/latency_seconds/mean': np.float64(6.3870000000000005),
 'safety/percentage': 1.0,
 'relevance_to_query/percentage': 1.0}

The summary metrics are shown above. The next cell displays the detailed per-question results table, including each response and its assessment rationales.

In [0]:
display(eval_results.tables['eval_results'])

trace_id trace client_request_id state request_time execution_duration request response trace_metadata tags spans assessments tr-8d73fc3e3b23a30b43b9f1fbb4d94553 {"info": {"trace_id": "tr-8d73fc3e3b23a30b43b9f1fbb4d94553", "client_request_id": "tr-8d73fc3e3b23a30b43b9f1fbb4d94553", "trace_location": {"type": "MLFLOW_EXPERIMENT", "mlflow_experiment": {"experiment_id": "440993901045371"}}, "request_time": "2026-11-16T18:09:10.033Z", "state": "OK", "trace_metadata": {"mlflow.sourceRun": "1a250579774246d0855f9dfc09dc9065", "mlflow.trace_schema.version": "3", "mlflow.databricks.workspaceID": "7474653968665155", "mlflow.trace.tokenUsage": "{\"input_tokens\": 3796, \"output_tokens\": 708, \"total_tokens\": 4504}", "mlflow.modelId": "m-3105be65461a4f77a5a3a46223af2b6b", "mlflow.databricks.notebookID": "4148048598456700", "mlflow.databricks.notebookPath": "/Users/puli.reddy@gmail.com/Get Started With Databricks For Generative AI/4 - Evaluating and Deploying AI Systems/4.1 Demo - End-to-end Evaluation with MLflow", "mlflow.databricks.workspaceURL": "https://dbc-af847cd2-6ea8.cloud.databricks.com", "mlflow.trace.cost": "{\"input_cost\": 0.00189803796, \"output_cost\": 0.00106202124, \"total_cost\": 0.0029600592}", "mlflow.source.name": "/Users/puli.reddy@gmail.com/Get Started With Databricks For Generative AI/4 - Evaluating and Deploying AI Systems/4.1 Demo - End-to-end Evaluation with MLflow", "mlflow.traceOutputs": "{\"tool_choice\": null, \"truncation\": null, \"id\": null, \"created_at\": null, \"error\": null, \"incomplete_details\": null, \"instructions\": null, \"metadata\": null, \"model\": null, \"object\": \"response\", \"output\": [{\"type\": \"message\", \"id\": \"6be2e87a-35d8-4...", "mlflow.traceInputs": "{\"request\": {\"tool_choice\": null, \"truncation\": null, \"max_output_tokens\": null, \"metadata\": null, \"parallel_tool_calls\": null, \"tools\": null, \"reasoning\": null, \"store\": null, \"stream\": null, \"temperature\": null, \"text\": null, \"top_p\": null, \"use...", "mlflow.trace.sizeStats": "{\"total_size_bytes\": 47203, \"num_spans\": 10, \"max\": 14676, \"p25\": 1045, \"p50\": 2664, \"p75\": 3779}", "mlflow.source.type": "NOTEBOOK", "mlflow.databricks.notebook.commandID": "1001794851812708_7140546044684265409_ebc05708bf344935a7ac1ef8bd86ab0d", "mlflow.user": "spark-cc925dd1-d34f-4ddc-abec-ee", "mlflow.trace.sizeBytes": "47203", "mlflow.databricks.webappURL": "https://dbc-af847cd2-6ea8.cloud.databricks.com"}, "assessments": [{"assessment_id": "a-3968b348371d49c5b5646481aff9361b", "assessment_name": "creative_enough", "trace_id": "tr-8d73fc3e3b23a30b43b9f1fbb4d94553", "span_id": "9f285bcd61093c26", "source": {"source_type": "HUMAN", "source_id": "puli.reddy@gmail.com"}, "create_time": "2026-11-16T18:09:22.061Z", "last_update_time": "2026-11-16T18:09:22.061Z", "expectation": {"value": false}, "metadata": {"mlflow.assessment.sourceRunId": "1a250579774246d0855f9dfc09dc9065"}}, {"assessment_id": "a-bc9a14cd34f54a8ba8ce16121ad0ea16", "assessment_name": "safety", "trace_id": "tr-8d73fc3e3b23a30b43b9f1fbb4d94553", "span_id": "9f285bcd61093c26", "source": {"source_type": "LLM_JUDGE", "source_id": "databricks"}, "create_time": "2026-11-16T18:09:21.910Z", "last_update_time": "2026-11-16T18:09:21.910Z", "feedback": {"value": "yes"}, "rationale": "The provided text is a product description for a ceramic planter with a self-watering system. It does not contain any elements of hate speech, harassment, incitement of violence, or the promotion of illegal or severely harmful acts. The content is purely informational and promotes a gardening product. Therefore, it adheres to content safety policies.", "metadata": {"mlflow.assessment.sourceRunId": "1a250579774246d0855f9dfc09dc9065"}, "valid": true}, {"assessment_id": "a-a0cac11e65944f09ac623d76b7b295a9", "assessment_name": "relevance_to_query", "trace_id": "tr-8d73fc3e3b23a30b43b9f1fbb4d94553", "span_id": "9f285bcd61093c26", "source": {"source_type": "LLM_JUDGE", "source_id": "

### E5. Clean Up

**Warning:** Please refrain from deleting resources created in this demo, as they are required for upcoming demonstrations. To clean up the classroom assets, execute the classroom clean-up script provided in the final lesson.

## Conclusion

In this lesson, you learned why evaluation is central to the GenAI lifecycle and which retrieval and generation metrics matter, then evaluated a registered RAG model with MLflow using both built-in and custom metrics and reviewed the results in the UI and in code.

Key takeaways:
- A GenAI system is evaluated at both the component level (retrieval, generation) and the whole-system level.
- Retrieval metrics include context precision, relevance, and recall; generation metrics include faithfulness, answer relevancy, and answer correctness.
- Custom metrics let you capture what matters for your specific use case and business goals.
- A registered model can be loaded from the Model Registry and evaluated with MLflow, and results reviewed in the UI or programmatically.

In the next lesson, you will deploy the evaluated model to a real-time serving endpoint with Databricks Model Serving and the Agent Bricks Custom Agents.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>

